EDA Dataset A, B, C

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import os

# Buat folder untuk menyimpan prediksi dan model jika belum ada
os.makedirs('predictions', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Definisikan daftar dataset yang akan diproses
datasets = {
    'A': {'train': 'train.csv', 'test': 'test.csv', 'id_tim': 'T017'}, # Ganti T001 dengan ID Tim kamu yang sebenarnya
    'B': {'train': 'train_B.csv', 'test': 'test_B.csv', 'id_tim': 'T017'},
    'C': {'train': 'train_C.csv', 'test': 'test_C.csv', 'id_tim': 'T017'}
}

# Loop untuk memproses setiap dataset secara otomatis
for key, paths in datasets.items():
    print(f"\n==============================")
    print(f"PROSES MODEL UNTUK DATASET {key}")
    print(f"==============================")

    # 1. Load Data
    train_df = pd.read_csv(paths['train'])
    test_df = pd.read_csv(paths['test'])

    print(f"Shape Train {key}: {train_df.shape}")
    print(f"Shape Test {key}: {test_df.shape}")

    # 2. Preprocessing & Split Data (Mencegah Data Leakage)
    # PENTING: sample_id tidak boleh jadi fitur model!
    X = train_df.drop(columns=['sample_id', 'target'])
    y = train_df['target']

    X_test = test_df.drop(columns=['sample_id'])
    test_ids = test_df['sample_id']

    # Stratified split untuk validasi internal (80:20)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    # Scaling fitur (fit HANYA pada data training internal)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # 3. Arsitektur Neural Network (Baseline MLP)
    tf.keras.utils.set_random_seed(42)
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid') # Ubah ke softmax/jumlah kelas jika multiclass
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Early Stopping agar tidak overfitting
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    # 4. Pelatihan Model
    history = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    # 5. Evaluasi Validasi
    y_val_pred_prob = model.predict(X_val_scaled)
    y_val_pred = (y_val_pred_prob > 0.5).astype(int).flatten()
    val_acc = accuracy_score(y_val, y_val_pred)
    print(f"Validation Accuracy Dataset {key}: {val_acc:.4f}")

    # 6. Prediksi ke Data Test (untuk Submission)
    y_test_pred_prob = model.predict(X_test_scaled)
    y_test_pred = (y_test_pred_prob > 0.5).astype(int).flatten()

    # Buat DataFrame Submission sesuai format aturan
    submission = pd.DataFrame({
        'sample_id': test_ids,
        'target': y_test_pred
    })

    # Simpan file CSV dengan format penamaan: IDTIM_Dataset_Attempt.csv (contoh: T001_A_01.csv)
    filename = f"predictions/{paths['id_tim']}_{key}_01.csv"
    submission.to_csv(filename, index=False)
    print(f"File submission berhasil disimpan di: {filename}")


PROSES MODEL UNTUK DATASET A
Shape Train A: (36249, 10)
Shape Test A: (9063, 9)
Epoch 1/30


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


907/907 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7477 - loss: 0.5202 - val_accuracy: 0.7789 - val_loss: 0.4675
Epoch 2/30
907/907 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7732 - loss: 0.4823 - val_accuracy: 0.7868 - val_loss: 0.4600
Epoch 3/30
907/907 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7775 - loss: 0.4740 - val_accuracy: 0.7877 - val_loss: 0.4548
Epoch 4/30
907/907 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7802 - loss: 0.4717 - val_accuracy: 0.7906 - val_loss: 0.4529
Epoch 5/30
907/907 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7815 - loss: 0.4649 - val_accuracy: 0.7902 - val_loss: 0.4499
Epoch 6/30
907/907 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7827 - loss: 0.4644 - val_accuracy: 0.7930 - val_loss: 0.4480
Epoch 7/30
907/907 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7844 - loss: 0.4613 - val_accuracy: 0.7924 - val_loss: 0.4454
Epoch 8/30
907/907 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7858 - loss: 0.4593 - val_accuracy: 0.7941 - val_

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1667 - loss: -50061.9531 - val_accuracy: 0.1667 - val_loss: -199976.4062
Epoch 2/30
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.1667 - loss: -671786.0000 - val_accuracy: 0.1667 - val_loss: -1334132.2500
Epoch 3/30
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1667 - loss: -2423819.7500 - val_accuracy: 0.1667 - val_loss: -3743109.0000
Epoch 4/30
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1667 - loss: -5579738.0000 - val_accuracy: 0.1667 - val_loss: -7654422.0000
Epoch 5/30
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1667 - loss: -10318315.0000 - val_accuracy: 0.1667 - val_loss: -13308869.0000
Epoch 6/30
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.1667 - loss: -16986314.0000 - val_accuracy: 0.1667 - val_loss: -20994426.0000
Epoch 7/30
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1667 - loss: -25825944.0000 - val_accuracy: 0.1667 - val_loss: -30945010.0000
E

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


240/240 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8103 - loss: 0.4828 - val_accuracy: 0.8573 - val_loss: 0.4151
Epoch 2/30
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8438 - loss: 0.4191 - val_accuracy: 0.8562 - val_loss: 0.4057
Epoch 3/30
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8513 - loss: 0.4150 - val_accuracy: 0.8536 - val_loss: 0.4008
Epoch 4/30
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8487 - loss: 0.4109 - val_accuracy: 0.8568 - val_loss: 0.3979
Epoch 5/30
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8521 - loss: 0.4040 - val_accuracy: 0.8562 - val_loss: 0.3957
Epoch 6/30
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8517 - loss: 0.4034 - val_accuracy: 0.8568 - val_loss: 0.3944
Epoch 7/30
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8547 - loss: 0.4018 - val_accuracy: 0.8573 - val_loss: 0.3932
Epoch 8/30
240/240 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8531 - loss: 0.4009 - val_accuracy: 0.8573 - val_

#Menangani Dataset B
karena seluruh nilai prediksi pada kolom target didataset B bernilai 1 Semuanya (tidak ada angka 0) karena multikelas

In [ ]:
# Cek jumlah kelas pada dataset B
import pandas as pd

# Ganti dengan nama file train Dataset B kamu (misal train_2.csv atau train.csv yang sesuai)
train_b = pd.read_csv('train_B.csv') # Sesuaikan path file train B
print("Kelas unik pada target Dataset B:", train_b['target'].unique())
print("Jumlah kelas:", train_b['target'].nunique())

Kelas unik pada target Dataset B: [2 3 1 0 4 5]
Jumlah kelas: 6


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Load Data Dataset B
train_df = pd.read_csv('train_B.csv')
test_df = pd.read_csv('test_B.csv')

print("Shape Train B:", train_df.shape)
print("Shape Test B:", test_df.shape)

# Cek jumlah kelas unik
num_classes = train_df['target'].nunique()
print(f"Jumlah kelas target: {num_classes}")

# 2. Preprocessing & Split Data (Mencegah Data Leakage)
X = train_df.drop(columns=['sample_id', 'target'])
y = train_df['target']

X_test = test_df.drop(columns=['sample_id'])
test_ids = test_df['sample_id']

# Stratified split untuk validasi internal (80:20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scaling fitur (fit HANYA pada data training internal)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Simpan scaler jika diperlukan untuk reproduksi
import joblib
joblib.dump(scaler, 'models/scaler_b.pkl')

# 3. Arsitektur Neural Network untuk Multiclass (Softmax)
tf.keras.utils.set_random_seed(42)
model_b = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    # Output layer: neuron sejumlah kelas (num_classes), aktivasi softmax
    Dense(num_classes, activation='softmax')
])

model_b.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early Stopping agar tidak overfitting
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True
)

# 4. Pelatihan Model
history = model_b.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# 5. Evaluasi Validasi
y_val_pred_prob = model_b.predict(X_val_scaled)
y_val_pred = np.argmax(y_val_pred_prob, axis=1)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy Dataset B: {val_acc:.4f}")

# 6. Prediksi ke Data Test (untuk Submission)
y_test_pred_prob = model_b.predict(X_test_scaled)
y_test_pred = np.argmax(y_test_pred_prob, axis=1)

# Pastikan label prediksi mencakup rentang kelas yang sesuai
print("Distribusi prediksi test B:", pd.Series(y_test_pred).value_counts())

# 7. Buat DataFrame Submission
submission_b = pd.DataFrame({
    'sample_id': test_ids,
    'target': y_test_pred
})

# Simpan dengan format ID Tim kamu (misal T017_B_01.csv)
filename = "predictions/T017_B_01.csv"
submission_b.to_csv(filename, index=False)
print(f"File submission Dataset B berhasil diperbarui dan disimpan di: {filename}")

# Simpan model weights
model_b.save('models/model_b_best.keras')

Shape Train B: (63988, 35)
Shape Test B: (15998, 34)
Jumlah kelas target: 6
Epoch 1/40


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1600/1600 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.5168 - loss: 1.1428 - val_accuracy: 0.5580 - val_loss: 1.0270
Epoch 2/40
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5455 - loss: 1.0523 - val_accuracy: 0.5633 - val_loss: 1.0134
Epoch 3/40
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5545 - loss: 1.0341 - val_accuracy: 0.5631 - val_loss: 1.0054
Epoch 4/40
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5564 - loss: 1.0281 - val_accuracy: 0.5669 - val_loss: 1.0007
Epoch 5/40
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5592 - loss: 1.0194 - val_accuracy: 0.5679 - val_loss: 0.9969
Epoch 6/40
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5576 - loss: 1.0156 - val_accuracy: 0.5706 - val_loss: 0.9937
Epoch 7/40
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5597 - loss: 1.0122 - val_accuracy: 0.5702 - val_loss: 0.9914
Epoch 8/40
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5624 - loss: 1.0107 - val_accurac

#Eksperimen Model

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import os
import time
import joblib

# Buat folder penyimpanan
os.makedirs('predictions', exist_ok=True)
os.makedirs('models', exist_ok=True)

In [ ]:
# Inisialisasi atau buat file experiments.csv
exp_file = 'experiments.csv'
if not os.path.exists(exp_file):
    pd.DataFrame(columns=[
        'dataset', 'config_name', 'seed', 'hidden_layers', 'dropout',
        'learning_rate', 'batch_size', 'val_accuracy', 'val_error', 'training_time_sec'
    ]).to_csv(exp_file, index=False)

In [ ]:
# Konfigurasi dataset
datasets_config = {
    'A': {'train': 'train.csv', 'test': 'test.csv', 'id_tim': 'T017', 'type': 'binary'},
    'B': {'train': 'train_B.csv', 'test': 'test_B.csv', 'id_tim': 'T017', 'type': 'multiclass'},
    'C': {'train': 'train_C.csv', 'test': 'test_C.csv', 'id_tim': 'T017', 'type': 'binary'} # Sesuaikan jika C multiclass
}

In [ ]:
# Definisi 2 Eksperimen per Dataset sesuai peraturan (Minimal 2 konfigurasi)
# Konfigurasi 1: Baseline (2 Hidden Layers)
# Konfigurasi 2: Model Lebih Dalam / Tuned (3 Hidden Layers + Variasi Dropout & LR)

for key, cfg in datasets_config.items():
    print(f"\n==========================================")
    print(f" MENJALANKAN EKSPERIMEN UNTUK DATASET {key} ")
    print(f"==========================================")

    train_df = pd.read_csv(cfg['train'])
    test_df = pd.read_csv(cfg['test'])

    X = train_df.drop(columns=['sample_id', 'target'])
    y = train_df['target']
    X_test = test_df.drop(columns=['sample_id'])
    test_ids = test_df['sample_id']

    num_classes = y.nunique()
    is_multiclass = (cfg['type'] == 'multiclass' or num_classes > 2)

    # Stratified Split (80:20) dengan Seed Tetap
    seed_val = 42
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.20, random_state=seed_val, stratify=y
    )

    # Preprocessing (Pencegahan Data Leakage: Fit Scaler HANYA pada X_train)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Simpan scaler untuk keperluan reproduksi laporan akhir
    joblib.dump(scaler, f"models/scaler_{key.lower()}.pkl")

    # Daftar 2 Eksperimen Konfigurasi
    experiments = [
        {
            'name': 'Config_1_Baseline',
            'layers': [64, 32],
            'dropout': 0.3,
            'lr': 0.001,
            'batch_size': 32
        },
        {
            'name': 'Config_2_Deep_Tuned',
            'layers': [128, 64, 32],
            'dropout': 0.2,
            'lr': 0.0005,
            'batch_size': 64
        }
    ]

    for idx, exp in enumerate(experiments, start=1):
        print(f"\n--- Menjalankan {exp['name']} untuk Dataset {key} ---")
        tf.keras.utils.set_random_seed(seed_val)

        # Bangun Arsitektur Neural Network
        model = Sequential()
        model.add(Dense(exp['layers'][0], activation='relu', input_shape=(X_train_scaled.shape[1],)))
        model.add(Dropout(exp['dropout']))

        for units in exp['layers'][1:]:
            model.add(Dense(units, activation='relu'))
            model.add(Dropout(exp['dropout']))

        if is_multiclass:
            model.add(Dense(num_classes, activation='softmax'))
            loss_fn = 'sparse_categorical_crossentropy'
        else:
            model.add(Dense(1, activation='sigmoid'))
            loss_fn = 'binary_crossentropy'

        optimizer = tf.keras.optimizers.Adam(learning_rate=exp['lr'])
        model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=6, restore_best_weights=True
        )

        start_time = time.time()
        history = model.fit(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            epochs=40,
            batch_size=exp['batch_size'],
            callbacks=[early_stop],
            verbose=0
        )
        train_time = time.time() - start_time


 MENJALANKAN EKSPERIMEN UNTUK DATASET A 

--- Menjalankan Config_1_Baseline untuk Dataset A ---


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Menjalankan Config_2_Deep_Tuned untuk Dataset A ---

 MENJALANKAN EKSPERIMEN UNTUK DATASET B 

--- Menjalankan Config_1_Baseline untuk Dataset B ---


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Menjalankan Config_2_Deep_Tuned untuk Dataset B ---

 MENJALANKAN EKSPERIMEN UNTUK DATASET C 

--- Menjalankan Config_1_Baseline untuk Dataset C ---


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Menjalankan Config_2_Deep_Tuned untuk Dataset C ---


In [ ]:
# Evaluasi Validasi
val_preds_prob = model.predict(X_val_scaled, verbose=0)
if is_multiclass:
    val_preds = np.argmax(val_preds_prob, axis=1)
else:
    val_preds = (val_preds_prob > 0.5).astype(int).flatten()

val_acc = accuracy_score(y_val, val_preds)
val_err = 1.0 - val_acc

print(f"Hasil {exp['name']} -> Val Accuracy: {val_acc:.4f} | Val Error: {val_err:.4f} | Waktu: {train_time:.2f}s")

# Rekap ke experiments.csv
new_row = pd.DataFrame([{
    'dataset': key,
    'config_name': exp['name'],
    'seed': seed_val,
    'hidden_layers': str(exp['layers']),
    'dropout': exp['dropout'],
    'learning_rate': exp['lr'],
    'batch_size': exp['batch_size'],
    'val_accuracy': round(val_acc, 5),
    'val_error': round(val_err, 5),
    'training_time_sec': round(train_time, 2)
}])
new_row.to_csv(exp_file, mode='a', header=False, index=False)

# Jika ini adalah konfigurasi terbaik atau attempt kedua, simpan sebagai submission kedua (misal: T017_A_02.csv)
if idx == 2:
    test_preds_prob = model.predict(X_test_scaled, verbose=0)
    if is_multiclass:
        test_preds = np.argmax(test_preds_prob, axis=1)
    else:
        test_preds = (test_preds_prob > 0.5).astype(int).flatten()

    sub_df = pd.DataFrame({'sample_id': test_ids, 'target': test_preds})
    sub_filename = f"predictions/{cfg['id_tim']}_{key}_02.csv"
    sub_df.to_csv(sub_filename, index=False)

    # Simpan model terbaik
    model.save(f"models/model_{key.lower()}_best.keras")
    print(f"File submission attempt 2 disimpan di: {sub_filename}")

print("\n=== SEMUA EKSPERIMEN SELESAI DAN TERCATAT DI experiments.csv ===")

Hasil Config_2_Deep_Tuned -> Val Accuracy: 0.8562 | Val Error: 0.1438 | Waktu: 10.98s
File submission attempt 2 disimpan di: predictions/T017_C_02.csv

=== SEMUA EKSPERIMEN SELESAI DAN TERCATAT DI experiments.csv ===


In [ ]:
import os
import time
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import joblib

# Tahap 1: Membuat folder direktori penyimpanan untuk hasil prediksi dan model
os.makedirs('predictions', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Tahap 2: Pemetaan file dataset secara eksplisit untuk Dataset A, B, dan C
datasets_config = {
    'A': {'train': 'train.csv', 'test': 'test.csv', 'id_tim': 'T017', 'type': 'binary'},
    'B': {'train': 'train_B.csv', 'test': 'test_B.csv', 'id_tim': 'T017', 'type': 'multiclass'},
    'C': {'train': 'train_C.csv', 'test': 'test_C.csv', 'id_tim': 'T017', 'type': 'binary'}
}

# Tahap 3: Inisialisasi file CSV untuk merekam rekapitulasi eksperimen (experiments.csv)
exp_file = 'experiments.csv'
pd.DataFrame(columns=[
    'dataset', 'config_name', 'seed', 'hidden_layers', 'dropout',
    'learning_rate', 'batch_size', 'val_accuracy', 'val_error', 'training_time_sec'
]).to_csv(exp_file, index=False)

# Tahap 4: Looping utama untuk memproses setiap dataset (A, B, dan C) secara berurutan
for key, cfg in datasets_config.items():
    print(f"\n==========================================")
    print(f" MENJALANKAN EKSPERIMEN UNTUK DATASET {key} ")
    print(f"==========================================")

    # 4.1: Memuat data latih dan data uji dari file CSV
    train_df = pd.read_csv(cfg['train'])
    test_df = pd.read_csv(cfg['test'])

    # 4.2: Memisahkan fitur (X) dan label target (y), serta menyimpan sample_id untuk data uji
    X = train_df.drop(columns=['sample_id', 'target'])
    y = train_df['target']
    X_test = test_df.drop(columns=['sample_id'])
    test_ids = test_df['sample_id']

    # 4.3: Menentukan apakah tugas klasifikasi bersifat biner atau multiclass
    num_classes = y.nunique()
    is_multiclass = (cfg['type'] == 'multiclass' or num_classes > 2)

    # 4.4: Membagi data latih menjadi data latih internal dan data validasi (80:20) dengan Stratified Split dan Random Seed tetap
    seed_val = 42
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.20, random_state=seed_val, stratify=y
    )

    # 4.5: Preprocessing - Melakukan scaling fitur (StandardScaler) HANYA pada data latih untuk mencegah data leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Menyimpan objek scaler ke folder models/ untuk reproduksi di masa mendatang
    joblib.dump(scaler, f"models/scaler_{key.lower()}.pkl")

    # Tahap 5: Mendefinisikan minimal 2 konfigurasi eksperimen per dataset (Baseline & Tuned)
    experiments = [
        {
            'name': 'Config_1_Baseline',
            'layers': [64, 32],
            'dropout': 0.3,
            'lr': 0.001,
            'batch_size': 32
        },
        {
            'name': 'Config_2_Deep_Tuned',
            'layers': [128, 64, 32],
            'dropout': 0.2,
            'lr': 0.0005,
            'batch_size': 64
        }
    ]

    # Tahap 6: Looping untuk menjalankan setiap konfigurasi eksperimen
    for idx, exp in enumerate(experiments, start=1):
        print(f"\n--- Menjalankan {exp['name']} untuk Dataset {key} ---")
        tf.keras.utils.set_random_seed(seed_val)

        # 6.1: Membangun arsitektur model Neural Network (MLP) dengan Dropout
        model = Sequential()
        model.add(Input(shape=(X_train_scaled.shape[1],)))
        model.add(Dense(exp['layers'][0], activation='relu'))
        model.add(Dropout(exp['dropout']))

        for units in exp['layers'][1:]:
            model.add(Dense(units, activation='relu'))
            model.add(Dropout(exp['dropout']))

        # Menyesuaikan lapisan output dan fungsi loss berdasarkan jenis klasifikasi (biner vs multiclass)
        if is_multiclass:
            model.add(Dense(num_classes, activation='softmax'))
            loss_fn = 'sparse_categorical_crossentropy'
        else:
            model.add(Dense(1, activation='sigmoid'))
            loss_fn = 'binary_crossentropy'

        # 6.2: Kompilasi model dengan optimizer Adam dan learning rate yang ditentukan
        optimizer = tf.keras.optimizers.Adam(learning_rate=exp['lr'])
        model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

        # Mengatur Early Stopping untuk menghentikan pelatihan jika val_loss tidak membaik (mencegah overfitting)
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=6, restore_best_weights=True
        )

        # 6.3: Melatih model (Training) dan mengukur waktu proses
        start_time = time.time()
        history = model.fit(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            epochs=30,
            batch_size=exp['batch_size'],
            callbacks=[early_stop],
            verbose=0
        )
        train_time = time.time() - start_time

        # 6.4: Evaluasi model pada data validasi
        val_preds_prob = model.predict(X_val_scaled, verbose=0)
        if is_multiclass:
            val_preds = np.argmax(val_preds_prob, axis=1)
        else:
            val_preds = (val_preds_prob > 0.5).astype(int).flatten()

        val_acc = accuracy_score(y_val, val_preds)
        val_err = 1.0 - val_acc

        print(f"Hasil {exp['name']} -> Val Accuracy: {val_acc:.4f} | Val Error: {val_err:.4f} | Waktu: {train_time:.2f}s")

        # 6.5: Mencatat hasil metrik eksperimen ke dalam file experiments.csv
        new_row = pd.DataFrame([{
            'dataset': key,
            'config_name': exp['name'],
            'seed': seed_val,
            'hidden_layers': str(exp['layers']),
            'dropout': exp['dropout'],
            'learning_rate': exp['lr'],
            'batch_size': exp['batch_size'],
            'val_accuracy': round(val_acc, 5),
            'val_error': round(val_err, 5),
            'training_time_sec': round(train_time, 2)
        }])
        new_row.to_csv(exp_file, mode='a', header=False, index=False)

        # 6.6: Untuk eksperimen kedua (Config 2), generate file prediksi test dan simpan bobot model terbaik (.keras)
        if idx == 2:
            test_preds_prob = model.predict(X_test_scaled, verbose=0)
            if is_multiclass:
                test_preds = np.argmax(test_preds_prob, axis=1)
            else:
                test_preds = (test_preds_prob > 0.5).astype(int).flatten()

            sub_df = pd.DataFrame({'sample_id': test_ids, 'target': test_preds})
            sub_filename = f"predictions/{cfg['id_tim']}_{key}_02.csv"
            sub_df.to_csv(sub_filename, index=False)

            model.save(f"models/model_{key.lower()}_best.keras")
            print(f"File submission attempt 2 disimpan di: {sub_filename}")

print("\n=== SEMUA EKSPERIMEN SELESAI DENGAN FILE YANG BENAR! ===")
print("Isi folder predictions:", os.listdir('predictions'))
print("Isi folder models:", os.listdir('models'))


 MENJALANKAN EKSPERIMEN UNTUK DATASET A 

--- Menjalankan Config_1_Baseline untuk Dataset A ---
Hasil Config_1_Baseline -> Val Accuracy: 0.7999 | Val Error: 0.2001 | Waktu: 49.59s

--- Menjalankan Config_2_Deep_Tuned untuk Dataset A ---
Hasil Config_2_Deep_Tuned -> Val Accuracy: 0.8112 | Val Error: 0.1888 | Waktu: 28.66s
File submission attempt 2 disimpan di: predictions/T017_A_02.csv

 MENJALANKAN EKSPERIMEN UNTUK DATASET B 

--- Menjalankan Config_1_Baseline untuk Dataset B ---
Hasil Config_1_Baseline -> Val Accuracy: 0.5715 | Val Error: 0.4285 | Waktu: 83.24s

--- Menjalankan Config_2_Deep_Tuned untuk Dataset B ---
Hasil Config_2_Deep_Tuned -> Val Accuracy: 0.5863 | Val Error: 0.4137 | Waktu: 48.18s
File submission attempt 2 disimpan di: predictions/T017_B_02.csv

 MENJALANKAN EKSPERIMEN UNTUK DATASET C 

--- Menjalankan Config_1_Baseline untuk Dataset C ---
Hasil Config_1_Baseline -> Val Accuracy: 0.8573 | Val Error: 0.1427 | Waktu: 13.73s

--- Menjalankan Config_2_Deep_Tuned untu